In [1]:
from IPython.display import display
import time
import hmac
import hashlib
import urllib.request
from urllib.parse import quote
import json
import pandas as pd

In [5]:
API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"
BASE_URL = "https://api.statiz.co.kr/baseballApi"


str_cols = ["p_name", "p_nameEN", "t_code"]

int_cols = [
    "p_no","leagueType","year","p_position",
    "G","PA","ePA","AB","R","H","2B","3B","HR","TB",
    "RBI","SB","CS","BB","HP","IB","SO","GDP","SH","SF",
    "PO","Ass","E"
]

float_cols = [
    "WAROff","WARDef",
    "AVG","OBP","SLG","OPS","wRCplus","WAR",
    "BBK","BABIP","wOBA"
]


def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
    )


def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        secret.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def call_statiz_api(path: str, query: dict):
    method = "GET"
    normalized = normalize_query(query)
    timestamp = str(int(time.time()))
    payload = f"{method}|{path}|{normalized}|{timestamp}"
    signature = make_signature(SECRET, payload)

    url = f"{BASE_URL}/{path}"
    if normalized:
        url += f"?{normalized}"

    headers = {
        "X-API-KEY": API_KEY,
        "X-TIMESTAMP": timestamp,
        "X-SIGNATURE": signature,
    }

    req = urllib.request.Request(url, method=method, headers=headers)

    with urllib.request.urlopen(req, timeout=30) as resp:
        body = resp.read().decode("utf-8", errors="replace")

    return json.loads(body)


def convert_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    for c in str_cols:
        if c in df.columns:
            df[c] = df[c].astype("object")

    for c in int_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

    for c in float_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def reshape_player_season(json_data: dict) -> pd.DataFrame:
    basic_df = pd.DataFrame(json_data["basic"]["list"])
    deepen_df = pd.DataFrame(json_data["deepen"]["list"])
    fielding_df = pd.DataFrame(json_data["fielding"]["list"])

    if basic_df.empty:
        return pd.DataFrame()

    # deepen / fielding에 year가 없으므로 basic의 순서를 따른다고 가정
    if not deepen_df.empty:
        deepen_df["year"] = basic_df["year"].values

    if not fielding_df.empty:
        fielding_df["year"] = basic_df["year"].values

    merged_df = basic_df.copy()

    if not deepen_df.empty:
        merged_df = merged_df.merge(
            deepen_df[["year", "BBK", "BABIP", "wOBA"]],
            on="year",
            how="left"
        )

    if not fielding_df.empty:
        merged_df = merged_df.merge(
            fielding_df[["year", "PO", "Ass", "E"]],
            on="year",
            how="left"
        )

    merged_df = convert_dtypes(merged_df)
    return merged_df


def get_roster_pnos(date_list, code="1", t_code_list=None):
    """
    여러 날짜 / 여러 팀 기준으로 roster를 모아 unique p_no 확보
    t_code_list를 None으로 두면 직접 팀코드를 넣어야 함
    """
    all_rows = []

    for date in date_list:
        for t_code in t_code_list:
            try:
                json_data = call_statiz_api(
                    "prediction/playerRoster",
                    {"date": date, "code": code, "t_code": t_code}
                )

                # 응답 구조에 맞게 list 추출
                # 실제 응답 구조에 따라 아래 부분은 한번 확인 필요
                roster_list = json_data.get("list", [])
                if len(roster_list) == 0 and "data" in json_data:
                    roster_list = json_data["data"]

                temp_df = pd.DataFrame(roster_list)
                if not temp_df.empty:
                    temp_df["date"] = date
                    temp_df["t_code_query"] = t_code
                    all_rows.append(temp_df)

                print(f"[ROSTER OK] {date} / {t_code}")

            except Exception as e:
                print(f"[ROSTER FAIL] {date} / {t_code} / {e}")

    if not all_rows:
        return pd.DataFrame()

    roster_df = pd.concat(all_rows, ignore_index=True)

    # p_no가 있는 선수만
    roster_df = roster_df.dropna(subset=["p_no"]).copy()

    # 중복 제거
    roster_unique = roster_df.drop_duplicates(subset=["p_no"]).copy()

    return roster_unique


def collect_all_player_season(pno_df: pd.DataFrame):
    all_player_rows = []

    pno_list = pno_df["p_no"].dropna().astype(int).unique().tolist()

    for i, p_no in enumerate(pno_list, start=1):
        try:
            json_data = call_statiz_api(
                "prediction/playerSeason",
                {"p_no": str(p_no)}
            )

            if json_data.get("result_cd") != 100:
                print(f"[SEASON SKIP] p_no={p_no}, msg={json_data.get('result_msg')}")
                continue

            player_df = reshape_player_season(json_data)

            if not player_df.empty:
                all_player_rows.append(player_df)

            print(f"[SEASON OK] {i}/{len(pno_list)} p_no={p_no}")

            # 너무 빠르게 치지 않도록 약간 쉬기
            time.sleep(0.2)

        except Exception as e:
            print(f"[SEASON FAIL] p_no={p_no}, error={e}")

    if not all_player_rows:
        return pd.DataFrame()

    final_df = pd.concat(all_player_rows, ignore_index=True)
    final_df = convert_dtypes(final_df)

    return final_df

In [7]:
team_codes = ["1001", "2002", "3001", "5002", "6002", "7002", "9002", "10001", "11001", "12001"]
#               삼성,    KIA,   롯데,     LG,   두산,   한화,    SSG,    키움,      NC,     KT

date_list = [
    "2025-04-01",
    "2025-05-01",
    "2025-06-01",
    "2025-07-01",
    "2025-08-01",
    "2025-09-01"
]

In [8]:
roster_unique = get_roster_pnos(
    date_list=date_list,
    code="1",
    t_code_list=team_codes
)

print(roster_unique.head())
print(roster_unique.shape)

[ROSTER OK] 2025-04-01 / 1001
[ROSTER OK] 2025-04-01 / 2002
[ROSTER OK] 2025-04-01 / 3001
[ROSTER OK] 2025-04-01 / 5002
[ROSTER OK] 2025-04-01 / 6002
[ROSTER OK] 2025-04-01 / 7002
[ROSTER OK] 2025-04-01 / 9002
[ROSTER OK] 2025-04-01 / 10001
[ROSTER OK] 2025-04-01 / 11001
[ROSTER OK] 2025-04-01 / 12001
[ROSTER OK] 2025-05-01 / 1001
[ROSTER OK] 2025-05-01 / 2002
[ROSTER OK] 2025-05-01 / 3001
[ROSTER OK] 2025-05-01 / 5002
[ROSTER OK] 2025-05-01 / 6002
[ROSTER OK] 2025-05-01 / 7002
[ROSTER OK] 2025-05-01 / 9002
[ROSTER OK] 2025-05-01 / 10001
[ROSTER OK] 2025-05-01 / 11001
[ROSTER OK] 2025-05-01 / 12001
[ROSTER OK] 2025-06-01 / 1001
[ROSTER OK] 2025-06-01 / 2002
[ROSTER OK] 2025-06-01 / 3001
[ROSTER OK] 2025-06-01 / 5002
[ROSTER OK] 2025-06-01 / 6002
[ROSTER OK] 2025-06-01 / 7002
[ROSTER OK] 2025-06-01 / 9002
[ROSTER OK] 2025-06-01 / 10001
[ROSTER OK] 2025-06-01 / 11001
[ROSTER OK] 2025-06-01 / 12001
[ROSTER OK] 2025-07-01 / 1001
[ROSTER OK] 2025-07-01 / 2002
[ROSTER OK] 2025-07-01 / 3001
[

In [3]:
roster_unique

NameError: name 'roster_unique' is not defined

In [ ]:
all_player_season_df = collect_all_player_season(roster_unique)

print(all_player_season_df.head())
print(all_player_season_df.shape)
print(all_player_season_df.dtypes)

-------------------------

In [3]:
API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"
BASE_URL = "https://api.statiz.co.kr/baseballApi"

METHOD = "GET"
PATH   = "prediction/playerRoster"
QUERY  = {
    "date": "2025-08-31",
    "code": "1",
    "t_code": "6002"
}

def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
    )

def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        SECRET.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

normalized = normalize_query(QUERY)
timestamp = str(int(time.time()))

payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
signature = make_signature(SECRET, payload)

url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

headers = {
    "X-API-KEY": API_KEY,
    "X-TIMESTAMP": timestamp,
    "X-SIGNATURE": signature,
}

req = urllib.request.Request(url, method=METHOD, headers=headers)

with urllib.request.urlopen(req, timeout=30) as resp:
    body = resp.read().decode("utf-8")

json_data = json.loads(body)

display(json_data)
print(json_data.keys())

{'0': {'name': '김인태', 'p_no': 11098, 't_code': 6002, 'pj_date': '2025-07-26'},
 '1': {'name': '이영하', 'p_no': 12568, 't_code': 6002, 'pj_date': '2025-08-28'},
 '2': {'name': '박치국', 'p_no': 12850, 't_code': 6002, 'pj_date': '2025-08-19'},
 '3': {'name': '최원준', 'p_no': 12852, 't_code': 6002, 'pj_date': '2025-07-17'},
 '4': {'name': '곽빈', 'p_no': 13061, 't_code': 6002, 'pj_date': '2025-06-03'},
 '5': {'name': '정수빈', 'p_no': 10195, 't_code': 6002, 'pj_date': '2025-03-22'},
 '6': {'name': '홍건희', 'p_no': 10696, 't_code': 6002, 'pj_date': '2025-07-30'},
 '7': {'name': '조수행', 'p_no': 12524, 't_code': 6002, 'pj_date': '2025-07-17'},
 '8': {'name': '강승호', 'p_no': 11087, 't_code': 6002, 'pj_date': '2025-08-05'},
 '9': {'name': '박계범', 'p_no': 11198, 't_code': 6002, 'pj_date': '2025-07-01'},
 '10': {'name': '박신지', 'p_no': 13070, 't_code': 6002, 'pj_date': '2025-06-14'},
 '11': {'name': '이병헌', 'p_no': 15099, 't_code': 6002, 'pj_date': '2025-08-28'},
 '12': {'name': '이유찬', 'p_no': 12894, 't_code': 600

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', 'result_cd', 'result_msg', 'update_time'])


In [7]:
roster_rows = [
    v for k, v in json_data.items()
    if isinstance(v, dict) and k.isdigit()
]

roster_df = pd.DataFrame(roster_rows)
roster_df

,name,p_no,t_code,pj_date
0,김인태,11098,6002,2025-07-26
1,이영하,12568,6002,2025-08-28
2,박치국,12850,6002,2025-08-19
3,최원준,12852,6002,2025-07-17
4,곽빈,13061,6002,2025-06-03
5,정수빈,10195,6002,2025-03-22
6,홍건희,10696,6002,2025-07-30
7,조수행,12524,6002,2025-07-17
8,강승호,11087,6002,2025-08-05
9,박계범,11198,6002,2025-07-01


In [13]:
def call_statiz_api(path: str, query: dict):
    method = "GET"
    normalized = normalize_query(query)
    timestamp = str(int(time.time()))
    payload = f"{method}|{path}|{normalized}|{timestamp}"
    signature = make_signature(SECRET, payload)

    url = f"https://api.statiz.co.kr/baseballApi/{path}"
    if normalized:
        url += f"?{normalized}"

    headers = {
        "X-API-KEY": API_KEY,
        "X-TIMESTAMP": timestamp,
        "X-SIGNATURE": signature,
    }

    req = urllib.request.Request(url, method=method, headers=headers)

    with urllib.request.urlopen(req, timeout=30) as resp:
        body = resp.read().decode("utf-8")

    return json.loads(body)

In [14]:
def get_player_roster_df(date: str, code: str, t_code: str):
    json_data = call_statiz_api(
        "prediction/playerRoster",
        {"date": date, "code": code, "t_code": t_code}
    )

    roster_rows = [
        v for k, v in json_data.items()
        if isinstance(v, dict) and k.isdigit()
    ]

    roster_df = pd.DataFrame(roster_rows)
    return roster_df

In [15]:
def get_roster_unique(date_list, code="1", t_code_list=None):
    all_dfs = []

    for date in date_list:
        for t_code in t_code_list:
            try:
                temp_df = get_player_roster_df(date, code, t_code)

                if not temp_df.empty:
                    temp_df["query_date"] = date
                    temp_df["query_t_code"] = t_code
                    all_dfs.append(temp_df)

                print(f"[OK] date={date}, t_code={t_code}, rows={len(temp_df)}")

            except Exception as e:
                print(f"[FAIL] date={date}, t_code={t_code}, error={e}")

    if len(all_dfs) == 0:
        return pd.DataFrame()

    roster_df = pd.concat(all_dfs, ignore_index=True)

    roster_unique = (
        roster_df
        .dropna(subset=["p_no"])
        .drop_duplicates(subset=["p_no"])
        .copy()
    )

    return roster_unique

In [20]:
roster_df = pd.DataFrame(
    [v for k, v in json_data.items() if k.isdigit()]
)

display(roster_df.head())
print(roster_df.shape)

,name,p_no,t_code,pj_date
0,김재환,10643,6002,2025-04-22
1,김인태,11098,6002,2025-03-22
2,이영하,12568,6002,2025-03-22
3,박치국,12850,6002,2025-03-22
4,최원준,12852,6002,2025-03-22


(27, 4)


In [22]:
pno_list = roster_df["p_no"].dropna().astype(int).unique().tolist()

print(pno_list)
print(len(pno_list))

[10643, 11098, 12568, 12850, 12852, 10195, 12524, 11087, 11198, 14770, 11213, 14170, 13070, 15127, 10165, 13268, 12549, 16067, 16069, 15253, 16287, 16288, 16293, 16299, 16300, 16363, 10124]
27


In [23]:
def get_player_season_df(p_no):
    API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
    SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"
    METHOD = "GET"
    PATH   = "prediction/playerSeason"
    QUERY  = {"p_no": str(p_no)}

    normalized = normalize_query(QUERY)
    timestamp = str(int(time.time()))

    payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
    signature = make_signature(SECRET, payload)

    url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

    headers = {
        "X-API-KEY": API_KEY,
        "X-TIMESTAMP": timestamp,
        "X-SIGNATURE": signature,
    }

    req = urllib.request.Request(url, method=METHOD, headers=headers)

    with urllib.request.urlopen(req, timeout=30) as resp:
        body = resp.read().decode("utf-8")

    json_data = json.loads(body)

    # 실패 응답 방어
    if json_data.get("result_cd") != 100:
        print(f"[SKIP] p_no={p_no}, msg={json_data.get('result_msg')}")
        return pd.DataFrame()

    basic_df = pd.DataFrame(json_data["basic"]["list"])
    deepen_df = pd.DataFrame(json_data["deepen"]["list"])
    fielding_df = pd.DataFrame(json_data["fielding"]["list"])

    if basic_df.empty:
        return pd.DataFrame()

    # year 붙이기
    if not deepen_df.empty:
        deepen_df["year"] = basic_df["year"].values

    if not fielding_df.empty:
        fielding_df["year"] = basic_df["year"].values

    merged_df = basic_df.copy()

    if not deepen_df.empty:
        merged_df = merged_df.merge(
            deepen_df[["year", "BBK", "BABIP", "wOBA"]],
            on="year",
            how="left"
        )

    if not fielding_df.empty:
        merged_df = merged_df.merge(
            fielding_df[["year", "PO", "Ass", "E"]],
            on="year",
            how="left"
        )

    return merged_df

In [24]:
def apply_dtypes(merged_df):
    str_cols = ["p_name","p_nameEN","t_code"]

    int_cols = [
        "p_no","leagueType","year","p_position",
        "G","PA","ePA","AB","R","H","2B","3B","HR","TB",
        "RBI","SB","CS","BB","HP","IB","SO","GDP","SH","SF",
        "PO","Ass","E"
    ]

    float_cols = [
        "WAROff","WARDef",
        "AVG","OBP","SLG","OPS","wRCplus","WAR",
        "BBK","BABIP","wOBA"
    ]

    merged_df = merged_df.copy()

    merged_df[str_cols] = merged_df[str_cols].astype("object")

    for c in int_cols:
        if c in merged_df.columns:
            merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce").astype("int64")

    for c in float_cols:
        if c in merged_df.columns:
            merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce")

    return merged_df

In [25]:
all_players = []

for i, p_no in enumerate(pno_list, start=1):
    try:
        temp_df = get_player_season_df(p_no)

        if not temp_df.empty:
            temp_df = apply_dtypes(temp_df)
            all_players.append(temp_df)

        print(f"[OK] {i}/{len(pno_list)} p_no={p_no}, rows={len(temp_df)}")

        time.sleep(0.2)

    except Exception as e:
        print(f"[FAIL] {i}/{len(pno_list)} p_no={p_no}, error={e}")

[OK] 1/27 p_no=10643, rows=3
[FAIL] 2/27 p_no=11098, error=Length of values (3) does not match length of index (6)
[FAIL] 3/27 p_no=12568, error='fielding'
[FAIL] 4/27 p_no=12850, error='fielding'
[FAIL] 5/27 p_no=12852, error='fielding'
[OK] 6/27 p_no=10195, rows=3
[FAIL] 7/27 p_no=12524, error=Length of values (3) does not match length of index (9)
[FAIL] 8/27 p_no=11087, error=Length of values (3) does not match length of index (7)
[FAIL] 9/27 p_no=11198, error=Length of values (3) does not match length of index (11)
[FAIL] 10/27 p_no=14770, error='fielding'
[OK] 11/27 p_no=11213, rows=3
[FAIL] 12/27 p_no=14170, error=Length of values (3) does not match length of index (8)
[FAIL] 13/27 p_no=13070, error='fielding'
[FAIL] 14/27 p_no=15127, error='fielding'
[OK] 15/27 p_no=10165, rows=3
[FAIL] 16/27 p_no=13268, error='fielding'
[OK] 17/27 p_no=12549, rows=3
[FAIL] 18/27 p_no=16067, error='fielding'
[OK] 19/27 p_no=16069, rows=2
[FAIL] 20/27 p_no=15253, error=Length of values (3) does 

In [30]:
len(all_players)

6

-----------------

In [32]:
roster_df

,name,p_no,t_code,pj_date
0,김재환,10643,6002,2025-04-22
1,김인태,11098,6002,2025-03-22
2,이영하,12568,6002,2025-03-22
3,박치국,12850,6002,2025-03-22
4,최원준,12852,6002,2025-03-22
5,정수빈,10195,6002,2025-03-22
6,조수행,12524,6002,2025-04-02
7,강승호,11087,6002,2025-03-22
8,박계범,11198,6002,2025-05-17
9,최승용,14770,6002,2025-05-27


In [ ]:
pitcher_list = []
hitter_list = []

for _, row in roster_df.iterrows():

    name = row["name"]
    p_no = int(row["p_no"])
    t_code = row["t_code"]
    pj_date = row["pj_date"]

    try:
        json_data = fetch_player_season_json(p_no)

        # API 실패 방어
        if json_data.get("result_cd") != 100:
            print(f"[SKIP] {name} ({p_no}) / API 실패")
            continue

        basic_df = pd.DataFrame(json_data["basic"]["list"])

        if basic_df.empty:
            print(f"[SKIP] {name} ({p_no}) / basic 비어있음")
            continue

        # 첫 행 기준으로 포지션 확인
        position = int(basic_df.iloc[0]["p_position"])

        player_info = {
            "name": name,
            "p_no": p_no,
            "t_code": t_code,
            "pj_date": pj_date,
            "p_position": position
        }

        if position == 1:
            pitcher_list.append(player_info)
            print(f"[PITCHER] {name} ({p_no})")
        else:
            hitter_list.append(player_info)
            print(f"[HITTER] {name} ({p_no})")

        time.sleep(0.2)

    except Exception as e:
        print(f"[FAIL] {name} ({p_no}) / {e}")